# Deploying Vision AI at the Edge
## Participant Colab: Hands-On Helmet Detection

**Workshop duration:** 2.5–3 hours  
**Required:** A laptop, browser and Google account  
**Special hardware:** Not required  

You will build and deploy this pipeline:

```text
Traffic video
    ↓
YOLO detection
    ↓
ByteTrack identities
    ↓
Rider–motorcycle–helmet association
    ↓
Multi-frame decision
    ↓
Annotated evidence video
    ↓
ONNX export → ONNX Runtime on CPU → benchmark
```

### Notebook labels

- 🟩 **RUN** — execute the cell without changing it.
- 👀 **OBSERVE** — inspect the output and answer the prompt.
- ✏️ **TRY** — change only the highlighted parameter.
- ⏭️ **OPTIONAL** — skip during the core workshop if time is short.

> In Google Colab, first select **File → Save a copy in Drive**. Run cells in order and stop at each checkpoint.

## Workshop milestones

The core path fits a 150-minute session. Each stage lists the time it is
budgeted and what can be dropped if the room is running behind.

| Stage | You will produce | Time | If behind, drop |
|---|---|---:|---|
| 1. Setup | Verified Colab environment | 15 min | — |
| 2. Detection | Annotated traffic frame | 15 min | Exercise 1 |
| 3. Tracking | Persistent rider IDs | 15 min | — |
| 4. Association | Rider–vehicle–helmet relationships | 25 min | the TRY experiment |
| 5. Decision | Pending/uncertain/final verdicts | 20 min | — |
| 6. Evidence | Annotated MP4 | 20 min | the download cell |
| 7. Edge runtime | ONNX model, CPU benchmark and report | 30 min | quantization discussion |
| 8. Review | Results and deployment discussion | 10 min | — |

Sections marked **Core** are run by everyone together. Sections marked
**Explore** are yours to complete after the workshop; skipping them costs
nothing later.


# 1. Setup — Core

*Budgeted: 15 min.*

The repository provides the trained model and a short sample video. The notebook implements the application logic step by step.

Before starting in Colab, select **Runtime → Change runtime type → T4 GPU** when available. CPU execution is also supported.

### Why versions and paths are made explicit

An edge pipeline must be reproducible. A model that works with one runtime version may export or execute differently with another. Centralising the model, video and output paths also makes the later move from Colab to Jetson straightforward.

In [ ]:
# 🟩 RUN — Install the workshop packages
# Install tested workshop dependencies.
# If Colab asks for a runtime restart after installation, restart and rerun from here.
!pip -q install "ultralytics>=8.3,<9" "onnx>=1.17,<2" "onnxruntime>=1.20,<2" "scipy>=1.13,<2"

In [ ]:
# 🟩 RUN — Download the workshop assets
from pathlib import Path
import os
import subprocess
import time
import urllib.request
import urllib.error

# The workshop needs exactly two files: the trained weights and one short
# clip. Fetching them directly transfers about 98 MB. Cloning the whole
# project repository would transfer several hundred MB of duplicated
# models and videos that this notebook never opens.
RAW_BASE = "https://raw.githubusercontent.com/EmertxeInfoTech/edge-ai-helmet-detection-workshop/main/final"

# Colab provides /content; locally the assets land beside this notebook.
PROJECT_ROOT = Path("/content/workshop") if Path("/content").exists() else Path("workshop")
MODEL_PATH = PROJECT_ROOT / "models" / "best.pt"
VIDEO_PATH = PROJECT_ROOT / "sample_videos" / "sample.mp4"
OUTPUT_DIR = PROJECT_ROOT / "workshop_output"

for directory in (MODEL_PATH.parent, VIDEO_PATH.parent, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def fetch(url, destination, expected_mb, attempts=2):
    """Download once. Rerunning this cell reuses a complete local file."""
    # A partial file from an interrupted download is smaller than expected,
    # so size is checked rather than mere existence.
    if destination.exists() and destination.stat().st_size > 0.9 * expected_mb * 1e6:
        print(f"present     {destination.name:<12} {destination.stat().st_size / 1e6:>6.1f} MB")
        return
    for attempt in range(1, attempts + 1):
        try:
            started = time.time()
            urllib.request.urlretrieve(url, destination)
            print(f"downloaded  {destination.name:<12} "
                  f"{destination.stat().st_size / 1e6:>6.1f} MB in {time.time() - started:.0f}s")
            return
        except urllib.error.URLError as error:
            print(f"attempt {attempt} failed: {error}")
    raise RuntimeError(f"Could not download {destination.name}. Ask the instructor for the USB copy.")


fetch(f"{RAW_BASE}/models/best.pt", MODEL_PATH, 67)
fetch(f"{RAW_BASE}/sample_videos/sample.mp4", VIDEO_PATH, 31)

print("\nModel: ", MODEL_PATH)
print("Video: ", VIDEO_PATH)
print("Output:", OUTPUT_DIR)


In [ ]:
# 🟩 RUN — Import libraries and identify the execution device
import platform
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import ultralytics

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Python:", platform.python_version())
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Execution device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### 🟩 RUN — Preflight check

This cell verifies all prerequisites before model execution. Do not continue until every item reports `OK`.

In [ ]:
# Confirm that the workshop can proceed before doing expensive model work.
import shutil
import onnxruntime as ort

preflight = {
    "trained model": MODEL_PATH.exists(),
    "sample video": VIDEO_PATH.exists(),
    "output directory": OUTPUT_DIR.exists() and os.access(OUTPUT_DIR, os.W_OK),
    "FFmpeg": shutil.which("ffmpeg") is not None,
    "ONNX Runtime": bool(ort.__version__),
}

for item, ready in preflight.items():
    print(f"{'OK' if ready else 'FAILED':<7} {item}")

assert all(preflight.values()), "Preflight failed. Read the troubleshooting section before continuing."
print("\n✅ Workshop environment is ready")

### Checkpoint 1

Expected result:

```text
OK      trained model
OK      sample video
OK      output directory
OK      FFmpeg
OK      ONNX Runtime
✅ Workshop environment is ready
```

Your CPU/GPU name and software versions may differ from those of the instructor.

# 2. First inference: what does the model see? — Core

*Budgeted: 15 min.*

The trained detector recognizes five application-specific classes:

| ID | Class | Role in the application |
|---:|---|---|
| 0 | rider | Person travelling on the motorcycle |
| 1 | motorcycle | Vehicle associated with the rider |
| 2 | helmet | Helmeted head |
| 3 | no_helmet | Unhelmeted head |
| 4 | license_plate | Candidate plate region |

Detection gives us objects. It does **not** tell us which rider, motorcycle, helmet and plate belong together.

In [ ]:
# 🟩 RUN — Load the model and run one-frame inference
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
CLASS_NAMES = {0: "rider", 1: "motorcycle", 2: "helmet", 3: "no_helmet", 4: "license_plate"}

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, 20)
ok, sample_frame = cap.read()
cap.release()
assert ok, "Could not read the sample video"

result = model.predict(sample_frame, conf=0.25, imgsz=640, device=DEVICE, verbose=False)[0]
print(f"Detected {len(result.boxes)} objects")
print("Frame shape:", sample_frame.shape)

In [ ]:
# 🟩 RUN — Display the annotated frame
def show_bgr(image, title="", figsize=(14, 8)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

show_bgr(result.plot(), "Custom YOLO inference on one traffic frame")

In [ ]:
# 🟩 RUN — Inspect classes, confidences and coordinates
print(f"{'class':<16} {'confidence':>10}  box [x1, y1, x2, y2]")
print("-" * 65)
for cls, conf, box in zip(result.boxes.cls, result.boxes.conf, result.boxes.xyxy):
    name = CLASS_NAMES[int(cls)]
    coords = [round(v, 1) for v in box.tolist()]
    print(f"{name:<16} {float(conf):>10.2f}  {coords}")

### Exercise 1 — Explore

Change `conf=0.25` to `0.50`, rerun the inference and record:

- How many detections disappeared?
- Did all removed detections look incorrect?
- Why is a higher threshold not automatically a better system?

### Checkpoint 2

You should be able to explain the difference between:

- a **class** — what the object is,
- a **bounding box** — where it is, and
- a **confidence score** — the detector's score for this observation.

### 👀 OBSERVE — Expected result

- The frame contains rider, motorcycle, helmet/no-helmet and plate boxes where detected.
- The printed table contains one row per detection.
- Exact counts and confidence values may vary with runtime/package versions.

Stop and ask for help if the model reports zero objects on the supplied sample frame.

# 3. Tracking: retaining identity across frames — Core

*Budgeted: 15 min.*

A detector treats every frame independently. A tracker adds an ID so that rider `27` in one frame can still be rider `27` in the next.

Tracking is essential because our final decision must be based on several observations—not one potentially blurred or occluded frame.

In [ ]:
# 🟩 RUN — Track objects and compare consecutive frames
tracking_model = YOLO(str(MODEL_PATH))
track_stream = tracking_model.track(
    source=str(VIDEO_PATH),
    tracker="bytetrack.yaml",
    persist=True,
    stream=True,
    classes=list(CLASS_NAMES),
    conf=0.25,
    imgsz=640,
    device=DEVICE,
    verbose=False,
)

tracked_results = []
for frame_no, tracked in enumerate(track_stream):
    if frame_no in (20, 21):
        tracked_results.append(tracked)
    if frame_no >= 21:
        break

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
for ax, tracked, frame_no in zip(axes, tracked_results, (20, 21)):
    ax.imshow(cv2.cvtColor(tracked.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {frame_no}: track IDs persist")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 🟩 RUN — Convert raw tracker output into readable dictionaries
def unpack_tracked_result(r):
    # Return {class_name: {track_id: box}} and matching confidences.
    detections = {name: {} for name in CLASS_NAMES.values()}
    confidences = {name: {} for name in CLASS_NAMES.values()}

    if r.boxes.id is None:
        return detections, confidences

    for cls, tid, box, conf in zip(
        r.boxes.cls.tolist(), r.boxes.id.tolist(),
        r.boxes.xyxy.tolist(), r.boxes.conf.tolist()
    ):
        name = CLASS_NAMES[int(cls)]
        detections[name][int(tid)] = box
        confidences[name][int(tid)] = float(conf)
    return detections, confidences

tracked_detections, tracked_confidences = unpack_tracked_result(tracked_results[0])
for name, objects in tracked_detections.items():
    print(f"{name:<15}: {len(objects):>2} objects — IDs {list(objects)}")

### Checkpoint 3

- Find one rider ID that appears in both displayed frames.
- What could cause its ID to change?
- Why is a tracking ID useful but not equivalent to a person's real identity?

# 4. Association: which objects belong together? — Core

*Budgeted: 25 min.*

Tracking gives identities, but the detections still form a flat list. The application must infer relationships:

1. rider ↔ motorcycle,
2. helmet/no-helmet ↔ rider,
3. licence plate ↔ motorcycle.

We formulate association as a minimum-cost assignment problem. The Hungarian algorithm finds the globally lowest-cost one-to-one assignment. We then reject any pairing whose cost is still unacceptable.

In [ ]:
# 🟩 RUN — Define geometric measurements
from scipy.optimize import linear_sum_assignment

def center(box):
    '''Return the geometric centre of [x1, y1, x2, y2].'''
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def top_center(box):
    '''Use the top of a rider box because that is where its head should be.'''
    x1, y1, x2, _ = box
    return ((x1 + x2) / 2, y1)

def bottom_center(box):
    '''Use the bottom of a motorcycle box as the expected plate region.'''
    x1, _, x2, y2 = box
    return ((x1 + x2) / 2, y2)

def iou(box_a, box_b):
    '''Intersection over Union: shared area divided by combined area.'''
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union else 0.0

def point_distance(point_a, point_b):
    '''Euclidean distance in image pixels.'''
    return float(np.hypot(point_a[0] - point_b[0], point_a[1] - point_b[1]))

In [ ]:
# 🟩 RUN — Define rider–motorcycle and rider–head association
def associate_riders_to_motorcycles(riders, motorcycles, max_cost=0.90):
    '''Return {rider_id: motorcycle_id} using global minimum-cost assignment.'''
    # Use cost = 1 - IoU, so high overlap becomes a low cost. The Hungarian
    # algorithm always produces the best available assignment; max_cost then
    # rejects assignments that are still geometrically implausible.
    if not riders or not motorcycles:
        return {}
    rider_ids, motorcycle_ids = list(riders), list(motorcycles)
    costs = np.array([
        [1.0 - iou(riders[rid], motorcycles[mid]) for mid in motorcycle_ids]
        for rid in rider_ids
    ])
    rows, cols = linear_sum_assignment(costs)
    return {
        rider_ids[r]: motorcycle_ids[c]
        for r, c in zip(rows, cols)
        if costs[r, c] <= max_cost
    }

def associate_heads_to_riders(riders, helmets, no_helmets,
                               helmet_confs, no_helmet_confs,
                               max_distance=60):
    '''Return {rider_id: (helmet_status, detector_confidence)}.'''
    # Helmet and no-helmet boxes compete in the same candidate pool because a
    # rider should receive at most one head classification in a frame.
    heads = {
        **{hid: (box, "helmet", helmet_confs[hid]) for hid, box in helmets.items()},
        **{hid: (box, "no_helmet", no_helmet_confs[hid]) for hid, box in no_helmets.items()},
    }
    if not riders or not heads:
        return {}
    rider_ids, head_ids = list(riders), list(heads)
    costs = np.array([
        [point_distance(top_center(riders[rid]), center(heads[hid][0])) for hid in head_ids]
        for rid in rider_ids
    ])
    rows, cols = linear_sum_assignment(costs)
    matches = {}
    for r, c in zip(rows, cols):
        if costs[r, c] <= max_distance:
            _, status, confidence = heads[head_ids[c]]
            matches[rider_ids[r]] = (status, confidence)
    return matches

In [ ]:
# 🟩 RUN — Apply association to one frame
det, confs = unpack_tracked_result(tracked_results[0])
rider_motorcycle = associate_riders_to_motorcycles(det["rider"], det["motorcycle"])
head_rider = associate_heads_to_riders(
    det["rider"], det["helmet"], det["no_helmet"],
    confs["helmet"], confs["no_helmet"]
)

print("Rider → motorcycle")
for rider_id, motorcycle_id in rider_motorcycle.items():
    overlap = iou(det["rider"][rider_id], det["motorcycle"][motorcycle_id])
    print(f"  rider {rider_id:>4} → motorcycle {motorcycle_id:>4}  IoU={overlap:.2f}")

print("\nRider → head status")
for rider_id, (status, confidence) in head_rider.items():
    print(f"  rider {rider_id:>4} → {status:<10} confidence={confidence:.2f}")

In [ ]:
# 🟩 RUN — Visualise the accepted relationships
def draw_associations(image, detections, rider_motorcycle, head_rider):
    canvas = image.copy()
    for rider_id, rider_box in detections["rider"].items():
        x1, y1, x2, y2 = map(int, rider_box)
        status = head_rider.get(rider_id, ("unknown", 0))[0]
        color = (0, 0, 255) if status == "no_helmet" else (0, 200, 0) if status == "helmet" else (0, 165, 255)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
        cv2.putText(canvas, f"R{rider_id}: {status}", (x1, max(18, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        if rider_id in rider_motorcycle:
            motorcycle_id = rider_motorcycle[rider_id]
            rc = tuple(map(int, center(rider_box)))
            mc = tuple(map(int, center(detections["motorcycle"][motorcycle_id])))
            cv2.line(canvas, rc, mc, (255, 255, 0), 2)
    return canvas

associated_frame = draw_associations(
    tracked_results[0].orig_img, det, rider_motorcycle, head_rider
)
show_bgr(associated_frame, "Rider status and rider–motorcycle associations")

### ✏️ TRY — One controlled association experiment

In the call to `associate_riders_to_motorcycles`, change only `max_cost`:

```python
max_cost=1.0   # accepts even zero-overlap assignments
max_cost=0.5   # demands at least 50% IoU
```

Observe which setting creates wrong lines and which setting rejects valid relationships. Restore the default before continuing.

# 5. Temporal decision: evidence across frames — Core

*Budgeted: 20 min.*

A single frame may be blurred, occluded or misclassified. We therefore keep multiple helmet observations for each tracked rider.

A verdict requires all three conditions:

1. enough observations,
2. sufficient majority agreement, and
3. sufficient average detector confidence for the majority class.

The visualization uses four deliberately different states:

| State | Meaning | Display |
|---|---|---|
| `pending` | Fewer than the required observations | Orange |
| `uncertain` | Enough observations, but confidence/agreement is weak | Yellow |
| `compliant` | Helmet evidence is accepted | Green |
| `violation` | No-helmet evidence is accepted | Red |

Riders for whom no usable head evidence was collected are left unannotated. This avoids filling the video with misleading `UNKNOWN` boxes.

In [ ]:
# 🟩 RUN — Define explainable temporal decision logic
from collections import Counter, defaultdict

MIN_OBSERVATIONS = 3
MIN_AGREEMENT = 0.67
MIN_CONFIDENCE = 0.70

def decide_status(observations):
    '''Convert repeated detector observations into an explainable verdict.

    Each observation is a tuple: ("helmet" or "no_helmet", confidence).
    The function deliberately distinguishes evidence that is still accumulating
    (PENDING) from evidence that is sufficient but contradictory/weak
    (UNCERTAIN). Only strong, consistent evidence becomes a final verdict.
    '''
    if len(observations) < MIN_OBSERVATIONS:
        return {
            "verdict": "pending",
            "observations": len(observations),
            "reason": "collecting more observations",
        }

    labels = [label for label, _ in observations]
    majority_label, majority_count = Counter(labels).most_common(1)[0]
    majority_confidences = [conf for label, conf in observations if label == majority_label]
    agreement = majority_count / len(observations)
    detector_confidence = float(np.mean(majority_confidences))

    if agreement < MIN_AGREEMENT:
        verdict, reason = "uncertain", "insufficient temporal agreement"
    elif detector_confidence < MIN_CONFIDENCE:
        verdict, reason = "uncertain", "insufficient detector confidence"
    else:
        verdict = "violation" if majority_label == "no_helmet" else "compliant"
        reason = "accepted"

    return {
        "verdict": verdict,
        "majority": majority_label,
        "agreement": agreement,
        "detector_confidence": detector_confidence,
        "observations": len(observations),
        "reason": reason,
    }

In [ ]:
# 🟩 RUN — Test the logic with controlled observation sequences
test_cases = {
    "one strong frame": [("no_helmet", 0.95)],
    "consistent violation": [("no_helmet", 0.91), ("no_helmet", 0.88), ("no_helmet", 0.94)],
    "low-confidence violation": [("no_helmet", 0.55), ("no_helmet", 0.61), ("no_helmet", 0.58)],
    "mixed evidence": [("no_helmet", 0.93), ("helmet", 0.85), ("no_helmet", 0.91)],
    "compliant": [("helmet", 0.89), ("helmet", 0.92), ("helmet", 0.87)],
}

for name, observations in test_cases.items():
    print(f"{name:<26} → {decide_status(observations)}")

### 👀 OBSERVE — Why four states?

| State | Interpretation |
|---|---|
| `PENDING` | More frames are required |
| `UNCERTAIN` | Evidence exists but is weak or contradictory |
| `COMPLIANT` | Helmet evidence passed the thresholds |
| `VIOLATION` | No-helmet evidence passed the thresholds |

The system is allowed to postpone or refuse a decision. That is essential when results may become enforcement evidence.

# 6. Complete pipeline and annotated evidence — Core

*Budgeted: 20 min.*

We now connect detection, tracking, association and temporal decisions. Every processed frame is annotated and written to a video.

The next cell processes the complete 500-frame clip, which takes roughly a minute on a Colab CPU session. The later frames matter: they carry most of the riders whose verdicts never firm up, and a video containing only confirmed violations would misrepresent what the system actually does. If the session is running behind, set `FAST_PREVIEW = True` in the cell to cap it at 150 frames.


In [ ]:
def draw_pipeline_frame(image, detections, rider_motorcycle, decisions):
    canvas = image.copy()
    colors = {
        "pending": (0, 165, 255),       # orange: more frames are needed
        "uncertain": (0, 215, 255),     # yellow: evidence exists but is weak
        "compliant": (0, 200, 0),       # green: helmet decision accepted
        "violation": (0, 0, 255),       # red: no-helmet decision accepted
    }

    # Motorcycles are supporting context, so use thin blue boxes. Drawing them
    # first keeps the rider verdict visually dominant where the boxes overlap.
    for motorcycle_id, box in detections["motorcycle"].items():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (255, 150, 0), 1)
        cv2.putText(canvas, f"M{motorcycle_id}", (x1, max(18, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 150, 0), 1)

    for rider_id, rider_box in detections["rider"].items():
        # A detector may see a rider without finding a head, or without pairing
        # that rider to a motorcycle. Such a box has no decision evidence yet.
        # Do not label it UNKNOWN: that creates clutter and suggests the system
        # evaluated the rider when it did not.
        if rider_id not in rider_motorcycle or rider_id not in decisions:
            continue

        decision = decisions[rider_id]
        verdict = decision["verdict"]
        color = colors[verdict]
        x1, y1, x2, y2 = map(int, rider_box)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 3 if verdict == "violation" else 2)

        label = f"R{rider_id} | {verdict.upper()}"
        if "agreement" in decision:
            label += f" | n={decision['observations']} a={decision['agreement']:.0%}"
        else:
            label += f" | n={decision['observations']}"
        cv2.putText(canvas, label, (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 2)

        if rider_id in rider_motorcycle:
            motorcycle_id = rider_motorcycle[rider_id]
            rider_point = tuple(map(int, center(rider_box)))
            motorcycle_point = tuple(map(int, center(detections["motorcycle"][motorcycle_id])))
            cv2.line(canvas, rider_point, motorcycle_point, (255, 255, 0), 2)

    return canvas

In [ ]:
# 🟩 RUN — Execute the complete application pipeline
pipeline_model = YOLO(str(MODEL_PATH))
observations_by_rider = defaultdict(list)
decisions_by_rider = {}
evidence = {}
# The sample clip is 500 frames (20 seconds). Processing all of it takes
# roughly a minute on a Colab CPU session and is worth the wait: the later
# frames carry most of the riders whose verdicts stay PENDING or UNCERTAIN,
# which is exactly what the annotated video is meant to show. Set
# FAST_PREVIEW = True only if the session is running behind schedule.
FAST_PREVIEW = False
FRAME_LIMIT = 150 if FAST_PREVIEW else None

source_info = cv2.VideoCapture(str(VIDEO_PATH))
SOURCE_FPS = source_info.get(cv2.CAP_PROP_FPS) or 25.0
SOURCE_WIDTH = int(source_info.get(cv2.CAP_PROP_FRAME_WIDTH))
SOURCE_HEIGHT = int(source_info.get(cv2.CAP_PROP_FRAME_HEIGHT))
source_info.release()

RAW_ANNOTATED_VIDEO = OUTPUT_DIR / "helmet_detection_annotated_raw.mp4"
video_writer = cv2.VideoWriter(
    str(RAW_ANNOTATED_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"),
    SOURCE_FPS, (SOURCE_WIDTH, SOURCE_HEIGHT)
)
assert video_writer.isOpened(), "Could not open the annotated-video writer"

pipeline_stream = pipeline_model.track(
    source=str(VIDEO_PATH), tracker="bytetrack.yaml", persist=True, stream=True,
    classes=list(CLASS_NAMES), conf=0.25, imgsz=640, device=DEVICE, verbose=False
)

# Counts frames actually annotated and written, which is what the
# summary below reports.
processed_frames = 0

for frame_no, r in enumerate(pipeline_stream):
    # Stop only when an explicit frame cap has been configured.
    if FRAME_LIMIT is not None and frame_no >= FRAME_LIMIT:
        break
    processed_frames = frame_no + 1

    frame_detections, frame_confidences = unpack_tracked_result(r)
    rider_motorcycle = associate_riders_to_motorcycles(
        frame_detections["rider"], frame_detections["motorcycle"]
    )
    head_rider = associate_heads_to_riders(
        frame_detections["rider"], frame_detections["helmet"],
        frame_detections["no_helmet"], frame_confidences["helmet"],
        frame_confidences["no_helmet"]
    )

    # Only retain head evidence for riders associated with a motorcycle.
    for rider_id, observation in head_rider.items():
        if rider_id not in rider_motorcycle:
            continue
        observations_by_rider[rider_id].append(observation)
        decision = decide_status(observations_by_rider[rider_id])
        decisions_by_rider[rider_id] = decision

        if decision["verdict"] == "violation" and rider_id not in evidence:
            evidence[rider_id] = {
                "frame": frame_no,
                "decision": decision,
                "motorcycle_id": rider_motorcycle[rider_id],
            }

    annotated = draw_pipeline_frame(
        r.orig_img, frame_detections, rider_motorcycle, decisions_by_rider
    )
    video_writer.write(annotated)

    # Store the first frame at which each violation is confirmed.
    for rider_id, item in evidence.items():
        if item["frame"] == frame_no and "image" not in item:
            item["image"] = annotated.copy()

video_writer.release()

print(f"Processed {processed_frames} frames")
print(f"Tracked riders with helmet observations: {len(observations_by_rider)}")
print(f"Confirmed violations: {len(evidence)}")
print("Raw annotated video:", RAW_ANNOTATED_VIDEO)

In [ ]:
# 🟩 RUN — Review and save violation snapshots
if evidence:
    fig, axes = plt.subplots(len(evidence), 1, figsize=(14, 7 * len(evidence)))
    axes = np.atleast_1d(axes)
    for ax, (rider_id, item) in zip(axes, evidence.items()):
        decision = item["decision"]
        ax.imshow(cv2.cvtColor(item["image"], cv2.COLOR_BGR2RGB))
        ax.set_title(
            f"Rider {rider_id} — frame {item['frame']} — "
            f"agreement {decision['agreement']:.0%}, "
            f"confidence {decision['detector_confidence']:.2f}"
        )
        ax.axis("off")
        output_path = OUTPUT_DIR / f"violation_rider_{rider_id}_frame_{item['frame']}.jpg"
        cv2.imwrite(str(output_path), item["image"])
    plt.tight_layout()
    plt.show()
else:
    print("No violation reached the configured evidence thresholds.")

In [ ]:
# 🟩 RUN — Show every verdict, including the ones that were withheld
from collections import Counter

print("Verdict distribution across all tracked riders")
for verdict, count in Counter(d["verdict"] for d in decisions_by_rider.values()).most_common():
    print(f"   {verdict:<10} {count}")

print(f"\n{'rider':>6} {'obs':>5} {'majority':>10} {'agree':>7} {'conf':>6}  verdict")
print("-" * 58)
# Most-observed riders first: those carry the most evidence.
for rider_id, decision in sorted(decisions_by_rider.items(),
                                 key=lambda kv: -kv[1]["observations"])[:12]:
    majority = decision.get("majority", "-")
    agreement = f"{decision['agreement']:.2f}" if "agreement" in decision else "-"
    confidence = f"{decision['detector_confidence']:.2f}" if "detector_confidence" in decision else "-"
    print(f"{rider_id:>6} {decision['observations']:>5} {majority:>10} "
          f"{agreement:>7} {confidence:>6}  {decision['verdict']}")

# The interesting cases: the detector mostly said "no helmet", yet the
# system declined to report a violation.
withheld = [(rid, d) for rid, d in decisions_by_rider.items()
            if d.get("majority") == "no_helmet" and d["verdict"] != "violation"]

print(f"\nRiders whose majority observation was no_helmet but who were NOT reported: {len(withheld)}")
for rider_id, decision in sorted(withheld, key=lambda kv: -kv[1]["observations"]):
    print(f"   rider {rider_id}: {decision['observations']} observations, "
          f"agreement {decision['agreement']:.2f}, confidence "
          f"{decision['detector_confidence']:.2f}  ->  {decision['reason']}")


### 👀 OBSERVE — The verdict the system refused to give

One rider in this clip is worth your attention. The detector called them *no helmet* in the large majority of well over a hundred frames — and the system still did **not** report a violation, because the averaged detector confidence landed just under the 0.70 bar.

Sit with that for a moment. On this evidence a human reviewer would likely say "that rider has no helmet." The pipeline declines to say it, because the rule it was given was not met.

You can move that bar. `MIN_CONFIDENCE = 0.65` converts this rider into a confirmed violation immediately. The question the workshop is really asking is not *can you* but *should you*:

- Every threshold you relax to catch this rider also admits every weaker case behind them.
- A false violation has a real-world cost: someone is wrongly accused on the strength of a blurred frame.
- A system that never abstains is not more accurate. It is only more confident.

Try the change, look at what else it lets through, and then decide what the right bar is. That judgement — not the detector — is the engineering.


## Preview and download the annotated video — Core

OpenCV writes the frames first. The next cell converts the result to H.264/YUV420p so it plays reliably in a browser and presentation software.

In [ ]:
# 🟩 RUN — Convert and preview the annotated video
from IPython.display import Video, display

ANNOTATED_VIDEO_PATH = OUTPUT_DIR / "helmet_detection_annotated.mp4"

# The source clip is 2560x1440, so the annotated copy is large. The preview
# is scaled to 720p at crf 26: small enough to embed in the notebook and
# to play reliably in a browser, while the full-resolution file stays on
# disk. The full 500-frame clip lands near 12 MB, comfortably inside the
# 20 MB embed guard below.
conversion = subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(RAW_ANNOTATED_VIDEO),
        "-vf", "scale=1280:-2",
        "-c:v", "libx264", "-preset", "fast", "-crf", "26",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(ANNOTATED_VIDEO_PATH),
    ],
    capture_output=True, text=True,
)

if conversion.returncode != 0:
    print("H.264 conversion was unavailable; falling back to the OpenCV output.")
    print(conversion.stderr[-500:])
    ANNOTATED_VIDEO_PATH = RAW_ANNOTATED_VIDEO


def preview_video(path, limit_mb=20):
    """Embed the video only when it is small enough to be safe.

    An embedded video is base64-encoded into the notebook itself, so a
    50 MB file becomes roughly 70 MB of output. That can exhaust the
    browser tab and is refused by some notebook front ends.
    """
    size_mb = path.stat().st_size / 1e6
    print(f"Annotated video: {path}")
    print(f"Size: {size_mb:.2f} MB")
    if size_mb > limit_mb:
        print(f"Too large to embed ({size_mb:.0f} MB > {limit_mb} MB). "
              "Download it from the file browser to watch it locally.")
        return
    display(Video(str(path), embed=True))


preview_video(ANNOTATED_VIDEO_PATH)


### 👀 OBSERVE — How many riders actually got a verdict?

Look at the printed counts. Far more riders were tracked than were judged, and only one became a violation.

That gap is the system working as designed. A rider glimpsed for two frames stays `PENDING`. A rider seen clearly but inconsistently stays `UNCERTAIN`. Neither is a failure, and neither is reported as a violation.

The next cell shows exactly which riders were withheld, and why.


### ⏭️ OPTIONAL — Download generated files

This is disabled by default so **Run all** does not unexpectedly open a browser download dialog.

In [ ]:
DOWNLOAD_OUTPUTS = False

if DOWNLOAD_OUTPUTS:
    from google.colab import files
    files.download(str(ANNOTATED_VIDEO_PATH))
else:
    print("Set DOWNLOAD_OUTPUTS = True when you are ready to download the MP4.")

# 7. Edge optimization: export to ONNX — Core

*Budgeted: 30 min.*

So far we have used the original PyTorch model. An edge deployment normally converts the model to a runtime-friendly representation.

ONNX provides a portable model graph that can be executed by ONNX Runtime on a laptop, Raspberry Pi-class Linux system or many accelerator toolchains. On NVIDIA Jetson, the next step is usually TensorRT.

### What “edge deployment in Colab” means

Colab is not the final edge device. It gives every participant a common environment in which to practise the same deployment steps:

1. freeze/export the trained model,
2. load it through an edge-oriented runtime,
3. force batch-one CPU inference,
4. measure latency and throughput,
5. check that conversion preserved useful detections, and
6. package the model plus a reproducible deployment report.

The resulting ONNX model and code can then be moved to an attendee's laptop, Raspberry Pi-class Linux device or another ONNX-supported target. Your Jetson demonstration adds TensorRT acceleration but is not required for participant exercises.

In [ ]:
# 🟩 RUN — Export the trained model to ONNX
export_model = YOLO(str(MODEL_PATH))

# Export a fixed-shape, batch-one graph. Fixed shapes are easier for many edge
# runtimes to optimise than completely dynamic input dimensions.
onnx_path = Path(export_model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=17,
))

print("Exported model:", onnx_path)
print(f"PyTorch size: {MODEL_PATH.stat().st_size / 1e6:.2f} MB")
print(f"ONNX size:    {onnx_path.stat().st_size / 1e6:.2f} MB")

## Benchmarking correctly

We compare PyTorch and ONNX on **the same CPU**. Comparing PyTorch on a T4 GPU with ONNX on a CPU would measure different hardware rather than runtime efficiency.

The first few runs are warm-up runs and are excluded. We report median and 95th-percentile latency rather than quoting one unusually fast measurement.

> Colab results characterize the Colab runtime—not a Jetson or Raspberry Pi. Benchmark again on the actual target device before making deployment claims.

In [ ]:
# 🟩 RUN — Compare PyTorch CPU and ONNX CPU fairly
import time

def synchronize_if_needed(device):
    # GPU kernels are asynchronous. Synchronisation ensures the timer includes
    # completed inference work. CPU execution is already synchronous.
    if device != "cpu" and torch.cuda.is_available():
        torch.cuda.synchronize()

def benchmark_yolo(yolo_model, frame, device, warmup=3, runs=15):
    '''Measure end-to-end batch-one latency through the public model API.'''
    # Warm-up removes model initialisation and memory-allocation overhead from
    # the measurements that represent steady-state operation.
    for _ in range(warmup):
        yolo_model.predict(frame, imgsz=640, conf=0.25, device=device, verbose=False)
    synchronize_if_needed(device)

    latencies_ms = []
    for _ in range(runs):
        start = time.perf_counter()
        yolo_model.predict(frame, imgsz=640, conf=0.25, device=device, verbose=False)
        synchronize_if_needed(device)
        latencies_ms.append((time.perf_counter() - start) * 1000)

    median_ms = float(np.median(latencies_ms))
    return {
        "median_ms": median_ms,
        "p95_ms": float(np.percentile(latencies_ms, 95)),
        "fps": 1000 / median_ms,
    }

# Create fresh runtime objects so neither side inherits state from an earlier
# tracking or prediction pass.
pt_cpu_model = YOLO(str(MODEL_PATH))
onnx_model = YOLO(str(onnx_path), task="detect")
pt_cpu_results = benchmark_yolo(pt_cpu_model, sample_frame, "cpu")
onnx_cpu_results = benchmark_yolo(onnx_model, sample_frame, "cpu")

print(f"{'runtime':<18} {'median latency':>17} {'p95 latency':>14} {'FPS':>10}")
print("-" * 63)
print(f"{'PyTorch / CPU':<18} {pt_cpu_results['median_ms']:>14.1f} ms {pt_cpu_results['p95_ms']:>11.1f} ms {pt_cpu_results['fps']:>10.1f}")
print(f"{'ONNX / CPU':<18} {onnx_cpu_results['median_ms']:>14.1f} ms {onnx_cpu_results['p95_ms']:>11.1f} ms {onnx_cpu_results['fps']:>10.1f}")

# A GPU number is useful as an instructor reference, but it is deliberately
# kept outside the fair CPU-to-CPU comparison above.
gpu_results = None
if torch.cuda.is_available():
    gpu_model = YOLO(str(MODEL_PATH))
    gpu_results = benchmark_yolo(gpu_model, sample_frame, 0)
    print(f"{'PyTorch / GPU':<18} {gpu_results['median_ms']:>14.1f} ms {gpu_results['p95_ms']:>11.1f} ms {gpu_results['fps']:>10.1f}")

### 👀 OBSERVE — What if ONNX is *slower*?

Compare your two rows. On many Colab CPU sessions ONNX Runtime comes out **slower** than PyTorch, and that is a genuine result rather than a broken export.

Export is a portability step, not an automatic speedup:

- PyTorch CPU inference already dispatches to heavily optimised kernels, so there is no large inefficiency for ONNX Runtime to remove.
- At batch size one there is little work to amortise graph-level optimisation over.
- The exported graph is still FP32. The gains usually attributed to "edge optimisation" come from the *next* steps — FP16 or INT8 precision, and a runtime such as TensorRT compiled for the specific device.

What export buys you here is a model that no longer depends on Python or PyTorch, and can be loaded by a runtime on a Jetson, a Raspberry Pi-class board or a phone. Measuring that model on the machine in front of you is what tells you whether it is fast enough.

> Record the number you actually measured in Exercise 4, including when it contradicts the expectation. A benchmark that only ever confirms the hoped-for answer is not a benchmark.


In [ ]:
# 🟩 RUN — Confirm that export preserved useful detections
# Visual sanity check: conversion must preserve useful detections.
# Use CPU on both sides so differences come from conversion/runtime rather than
# different numerical hardware paths.
pt_prediction = pt_cpu_model.predict(sample_frame, imgsz=640, conf=0.25, device="cpu", verbose=False)[0]
onnx_prediction = onnx_model.predict(sample_frame, imgsz=640, conf=0.25, device="cpu", verbose=False)[0]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
axes[0].imshow(cv2.cvtColor(pt_prediction.plot(), cv2.COLOR_BGR2RGB))
axes[0].set_title(f"PyTorch: {len(pt_prediction.boxes)} detections")
axes[1].imshow(cv2.cvtColor(onnx_prediction.plot(), cv2.COLOR_BGR2RGB))
axes[1].set_title(f"ONNX: {len(onnx_prediction.boxes)} detections")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Run the exported model as a CPU edge application — Core

This is the participant's deployable edge implementation. The input remains a video file, but inference now runs through the exported ONNX model on CPU with batch size one. The same pattern can later consume frames from a USB, CSI or IP camera.

For workshop pacing, `EDGE_FRAME_LIMIT` defaults to 100 frames. Set it to `None` after the session to process the complete clip through ONNX Runtime.

In [ ]:
# 🟩 RUN — Execute the ONNX model as a CPU edge runtime
# Number of frames for the attendee runtime-validation exercise.
# Set to None to process the entire video using ONNX Runtime on CPU.
EDGE_FRAME_LIMIT = 100
EDGE_RAW_VIDEO = OUTPUT_DIR / "edge_onnx_cpu_raw.mp4"
EDGE_VIDEO_PATH = OUTPUT_DIR / "edge_onnx_cpu.mp4"

edge_writer = cv2.VideoWriter(
    str(EDGE_RAW_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"),
    SOURCE_FPS, (SOURCE_WIDTH, SOURCE_HEIGHT)
)
assert edge_writer.isOpened(), "Could not open the ONNX edge-video writer"

edge_latencies_ms = []

# stream=True prevents the runtime from retaining all video results in memory.
edge_stream = onnx_model.predict(
    source=str(VIDEO_PATH), stream=True, imgsz=640, conf=0.25,
    device="cpu", verbose=False
)

for edge_frame_no, edge_result in enumerate(edge_stream):
    if EDGE_FRAME_LIMIT is not None and edge_frame_no >= EDGE_FRAME_LIMIT:
        break

    # Ultralytics reports preprocessing, inference and postprocessing times.
    # The sum is a more honest application-facing latency than inference alone.
    stage_times = edge_result.speed
    edge_latencies_ms.append(
        stage_times.get("preprocess", 0.0)
        + stage_times.get("inference", 0.0)
        + stage_times.get("postprocess", 0.0)
    )

    # result.plot() renders the detections produced by the ONNX runtime.
    edge_writer.write(edge_result.plot())

edge_writer.release()

edge_conversion = subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(EDGE_RAW_VIDEO),
        "-vf", "scale=1280:-2",
        "-c:v", "libx264", "-preset", "fast", "-crf", "26",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(EDGE_VIDEO_PATH),
    ],
    capture_output=True, text=True,
)
if edge_conversion.returncode != 0:
    EDGE_VIDEO_PATH = EDGE_RAW_VIDEO

edge_median_ms = float(np.median(edge_latencies_ms))
edge_p95_ms = float(np.percentile(edge_latencies_ms, 95))
print(f"Frames processed by ONNX Runtime: {len(edge_latencies_ms)}")
print(f"Median pipeline latency: {edge_median_ms:.1f} ms")
print(f"P95 pipeline latency: {edge_p95_ms:.1f} ms")
print(f"Approximate throughput: {1000 / edge_median_ms:.1f} FPS")
print("Output:", EDGE_VIDEO_PATH)
preview_video(EDGE_VIDEO_PATH)

## Create a deployment report — Core

A model file without provenance or measurements is difficult to reproduce. The report below records the model, runtime, environment and benchmark result that should travel with the exported artifact.

In [ ]:
# 🟩 RUN — Package runtime and benchmark metadata
import json
from datetime import datetime, timezone

deployment_report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "application": "helmet violation detection",
    "model": {
        "source": MODEL_PATH.name,
        "export": onnx_path.name,
        "input_size": 640,
        "source_size_mb": round(MODEL_PATH.stat().st_size / 1e6, 2),
        "onnx_size_mb": round(onnx_path.stat().st_size / 1e6, 2),
    },
    "runtime": {
        "name": "ONNX Runtime through Ultralytics",
        "device": "CPU",
        "batch_size": 1,
        "python": platform.python_version(),
        "ultralytics": ultralytics.__version__,
    },
    "single_frame_benchmark": onnx_cpu_results,
    "stream_validation": {
        "frames": len(edge_latencies_ms),
        "median_pipeline_ms": round(edge_median_ms, 2),
        "p95_pipeline_ms": round(edge_p95_ms, 2),
        "approx_fps": round(1000 / edge_median_ms, 2),
    },
    "limitations": [
        "Colab CPU results do not represent Jetson performance",
        "Accuracy must be validated on a labelled test set after conversion",
        "Camera capture, power and thermal behaviour are not measured in Colab",
    ],
}

REPORT_PATH = OUTPUT_DIR / "edge_deployment_report.json"
with open(REPORT_PATH, "w") as report_file:
    json.dump(deployment_report, report_file, indent=2)

print(json.dumps(deployment_report, indent=2))
print("Saved:", REPORT_PATH)

## Where quantization fits — Explore

Quantization reduces numerical precision, commonly from FP32 to FP16 or INT8. It can reduce memory and improve throughput, but it is not a universal one-line speedup:

- **Jetson:** TensorRT FP16 is the most practical instructor demonstration. INT8 additionally needs representative calibration data.
- **General CPU:** INT8 performance depends on the processor, supported operators and runtime. Quantizing without validating accuracy and operator support can make the model slower or unusable.
- **Colab workshop:** ONNX CPU deployment is the reliable common path. Quantization is discussed and can be demonstrated on your Jetson without making participant success depend on Jetson hardware.

This separation is intentional: every attendee completes a genuine export–runtime–benchmark–validation workflow, while the instructor shows the hardware-specific optimisation stage.

### Exercise 4 — Explore

Record your result:

| Metric | PyTorch | ONNX |
|---|---:|---:|
| Model size (MB) |  |  |
| Median latency (ms) |  |  |
| P95 latency (ms) |  |  |
| FPS |  |  |
| Detections in test frame |  |  |

Answer:

1. Which runtime was faster on your Colab session?
2. Did export change the detections?
3. Why is model size alone insufficient for selecting an edge runtime?

# Troubleshooting

| Symptom | Action |
|---|---|
| Asset download fails | Rerun the setup cell; it retries automatically, then ask the instructor for the USB copy |
| Model or video is missing | Rerun the setup cell and inspect the printed paths; a partial file is re-fetched automatically |
| Colab disconnects | Reconnect and rerun from Setup; generated runtime files are temporary |
| No GPU is available | Continue on CPU; GPU is optional for the participant path |
| Only one violation is confirmed | That is the correct result for this clip. Run the verdict-breakdown cell to see the riders who were withheld, and why |
| Video is shorter than the source | `FAST_PREVIEW` is `True`; set it to `False` and rerun the pipeline |
| MP4 does not preview | Confirm FFmpeg passed preflight; download the file and play it locally |
| ONNX export is slow | Wait for completion and do not repeatedly rerun the export cell |

If you become blocked during the workshop, move to the instructor's pre-executed backup notebook and continue with the interpretation exercises.

## Deployment checklist

- [ ] Target FPS and maximum end-to-end latency defined
- [ ] Camera resolution and lighting conditions tested
- [ ] Confidence, association and temporal thresholds validated
- [ ] Blur, occlusion, crowded scenes and low-light failures recorded
- [ ] Pending/uncertain evidence retained instead of forcing a classification
- [ ] Evidence image and decision metadata stored together
- [ ] Model/runtime versions recorded
- [ ] Accuracy checked again after conversion or quantization
- [ ] Sustained device temperature, power and memory measured
- [ ] Human review included before any enforcement action

# Completion checklist

- [ ] I ran custom YOLO inference on a traffic frame.
- [ ] I observed persistent tracking IDs.
- [ ] I associated riders, motorcycles and helmet status.
- [ ] I explained `PENDING`, `UNCERTAIN`, `COMPLIANT` and `VIOLATION`.
- [ ] I generated an annotated MP4.
- [ ] I exported the model to ONNX.
- [ ] I ran ONNX Runtime on CPU.
- [ ] I compared PyTorch CPU and ONNX CPU latency.
- [ ] I created `edge_deployment_report.json`.

## Takeaways

1. A detector produces objects; an application creates relationships and decisions.
2. Tracking and temporal evidence are essential for video decisions.
3. The best available association may still be unacceptable.
4. A responsible system can postpone or refuse a decision.
5. Edge deployment must be measured on the intended runtime and target device.

### Continue after the workshop

Try full-video processing, region-of-interest filtering, blur rejection, licence-plate cropping and—if you have access to suitable hardware—TensorRT FP16 deployment.